# Bayesian Optimization Capstone -- Week 7 Queries

All changes from Week 6 are marked with `# CHANGED:` or `# NEW:` comments.

### Summary of changes from Week 6
| Change | Week 6 | Week 7 | Reason |
|---|---|---|---|
| kappa for F6 | 2.0 | 2.5 | Badly regressed W6 to -1.499 |
| kappa for F7 | 2.5 | 1.0 | Near miss 1.926 vs best 1.958, begin exploiting |
| kappa for F5, F8 | 0.5 | 0.5 | Unchanged -- consistent gains |
| Week 6 data appended | No | Yes | Growing dataset |
| Seed | 456 | 789 | Rotate seed each round |

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.spatial.distance import cdist

# CHANGED: seed rotated to 789
np.random.seed(789)
print('Libraries loaded.')

## 2. Load All Historical Data (Initial + Weeks 1-6)

**CHANGED from Week 6:** Week 6 queries and outputs appended.

In [ ]:
DATA_PATH = './data/'

week1_queries={1:np.array([0.020584,0.969910]),2:np.array([0.813958,0.968718]),3:np.array([0.412118,0.126161,0.493019]),4:np.array([0.390847,0.455494,0.413276,0.470071]),5:np.array([0.342735,0.792788,0.998424,0.935381]),6:np.array([0.690974,0.164681,0.103053,0.037619,0.154198]),7:np.array([0.030065,0.557667,0.280418,0.193751,0.310746,0.755561]),8:np.array([0.190497,0.131345,0.026522,0.042568,0.730959,0.283019,0.005482,0.388835])}
week1_outputs={1:1.966e-321,2:-0.017659539446735338,3:-0.03594378093624019,4:-1.3242014729441993,5:2021.562348501267,6:-1.8227163856485316,7:1.3096626182809736,8:9.810010198309}
week2_queries={1:np.array([0.999999,0.999999]),2:np.array([0.673666,0.918591]),3:np.array([0.506031,0.565092,0.511916]),4:np.array([0.462210,0.470559,0.312877,0.405903]),5:np.array([0.505785,0.709925,0.994838,0.985391]),6:np.array([0.506031,0.565092,0.511916,0.972186,0.614903]),7:np.array([0.057378,0.366665,0.383868,0.136267,0.331203,0.752374]),8:np.array([0.032926,0.283604,0.006343,0.409730,0.814350,0.536043,0.151336,0.699855])}
week2_outputs={1:1.517648729565899e-192,2:0.510292287604992,3:-0.010612319162868315,4:-2.197214238202132,5:2201.3475945869486,6:-1.107214322524676,7:1.9584243277849602,8:9.8526837633915}
week3_queries={1:np.array([0.418744,0.463065]),2:np.array([0.718071,0.879689]),3:np.array([0.611094,0.382817,0.600071]),4:np.array([0.424666,0.520010,0.477146,0.381999]),5:np.array([0.524778,0.842229,0.984007,0.984075]),6:np.array([0.357049,0.223854,0.497338,0.808843,0.076793]),7:np.array([0.007196,0.287182,0.431297,0.103298,0.259892,0.771925]),8:np.array([0.024544,0.175956,0.116596,0.359046,0.449942,0.482790,0.135292,0.369405])}
week3_outputs={1:-0.005743923859916853,2:0.43088672102497183,3:-0.06351273264725116,4:-2.2609959093621828,5:2761.25406610152,6:-0.42557686218603197,7:1.7793279687424954,8:9.8684416967155}
week4_queries={1:np.array([0.050000,0.683166]),2:np.array([0.684422,0.983396]),3:np.array([0.773956,0.438878,0.858598]),4:np.array([0.437029,0.343930,0.415384,0.380827]),5:np.array([0.706600,0.999024,0.993655,0.986456]),6:np.array([0.400742,0.367525,0.463774,0.773392,0.094658]),7:np.array([0.085756,0.416199,0.486065,0.134693,0.401951,0.908800]),8:np.array([0.139770,0.331568,0.075811,0.049482,0.626413,0.402617,0.343854,0.516662])}
week4_outputs={1:1.1211575454297984e-144,2:0.45409752904570855,3:-0.03246674358598379,4:0.3657242066980797,5:5079.038688784203,6:-0.21379268800966483,7:1.178017048841175,8:9.8783244032591}
week5_queries={1:np.array([0.669598,0.050000]),2:np.array([0.735047,0.098026]),3:np.array([0.998632,0.651661,0.940995]),4:np.array([0.439093,0.344645,0.450032,0.381672]),5:np.array([0.934309,0.964839,0.991679,0.961572]),6:np.array([0.335944,0.499225,0.420450,0.716945,0.053954]),7:np.array([0.682352,0.053821,0.220360,0.184372,0.175906,0.812095]),8:np.array([0.185222,0.323353,0.012446,0.210805,0.688551,0.466240,0.073074,0.434930])}
week5_outputs={1:1.2526182543541078e-139,2:0.37763279019568097,3:-0.17975723176997382,4:-0.10027419641986723,5:6204.005487139041,6:-0.562862619219461,7:0.8855931591908454,8:9.8679736752075}
week6_queries={1:np.array([0.050000,0.135930]),2:np.array([0.673903,0.198094]),3:np.array([0.941407,0.229430,0.767542]),4:np.array([0.467922,0.285493,0.372897,0.437788]),5:np.array([0.979491,0.966487,0.975465,0.931802]),6:np.array([0.469673,0.447655,0.124057,0.754850,0.972988]),7:np.array([0.062617,0.306301,0.403156,0.127209,0.385557,0.539932]),8:np.array([0.119949,0.027514,0.106943,0.240757,0.633207,0.420766,0.184013,0.620722])}
# CHANGED: Week 6 outputs added
week6_outputs={1:-8.02263126364567e-154,2:0.5634168991004778,3:-0.1296499558422766,4:-1.8752576382494044,5:6210.693425379411,6:-1.4987906811967835,7:1.926382419159767,8:9.953627458159099}

# CHANGED: now includes 6 rounds of history
functions = {}
for i in range(1, 9):
    X = np.load(f'{DATA_PATH}function_{i}_initial_inputs.npy')
    y = np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy')
    X = np.vstack([X, week1_queries[i], week2_queries[i], week3_queries[i],
                   week4_queries[i], week5_queries[i], week6_queries[i]])
    y = np.append(y, [week1_outputs[i], week2_outputs[i], week3_outputs[i],
                      week4_outputs[i], week5_outputs[i], week6_outputs[i]])
    functions[i] = {'X': X, 'y': y}

print(f"{'Fn':<5} {'n pts':<8} {'All-time Best':<16} {'W6 Output':<16} {'W6 Improved?'}")
print('-' * 65)
for i in range(1, 9):
    prev = max(np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy').max(),
               week1_outputs[i], week2_outputs[i], week3_outputs[i],
               week4_outputs[i], week5_outputs[i])
    improved = week6_outputs[i] > prev
    print(f"F{i:<4} {len(functions[i]['y']):<8} {functions[i]['y'].max():<16.6f} "
          f"{week6_outputs[i]:<16.6f} {'YES' if improved else 'no'}")

## 3. Feature-Output Correlation Check

In [ ]:
print("Feature-output Pearson correlations (all data to date):\n")
for func_id, d in functions.items():
    X, y = d['X'], d['y']
    corrs = [round(float(np.corrcoef(X[:, j], y)[0, 1]), 3) for j in range(X.shape[1])]
    labels = [f'x{j+1}:{c}' for j, c in enumerate(corrs)]
    print(f"F{func_id}: {', '.join(labels)}")

## 4. Core GP and Acquisition Functions

Unchanged from Week 6. Seed updated to 789.

In [ ]:
def fit_gp(X, y):
    """Unchanged from prior weeks."""
    gp = GaussianProcessRegressor(
        kernel=Matern(nu=2.5), n_restarts_optimizer=10, normalize_y=True
    )
    gp.fit(X, y)
    return gp

def suggest_next_query(X, y, n_dims, n_candidates=100000, kappa=2.0, seed=789):
    """CHANGED: seed updated to 789."""
    rng = np.random.default_rng(seed)
    gp = fit_gp(X, y)
    candidates = rng.uniform(0, 1, size=(n_candidates, n_dims))
    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + kappa * sigma
    best_idx = np.argmax(ucb)
    best_x = candidates[best_idx]
    mu_b, sig_b = gp.predict(best_x.reshape(1, -1), return_std=True)
    return best_x, float(mu_b.ravel()[0]), float(sig_b.ravel()[0])

def max_distance_query(X_obs, resolution=200):
    """Unchanged from Week 3."""
    x1 = np.linspace(0.05, 0.95, resolution)
    x2 = np.linspace(0.05, 0.95, resolution)
    X1, X2 = np.meshgrid(x1, x2)
    grid = np.column_stack([X1.ravel(), X2.ravel()])
    dists = cdist(grid, X_obs)
    return grid[np.argmax(dists.min(axis=1))]

def format_query(x):
    return '-'.join([f'{v:.6f}' for v in np.clip(x, 0.0, 0.999999)])

print('Functions defined.')

## 5. Week 7 Adaptive Kappa Strategy

**CHANGED from Week 6:** F6 increased to 2.5 (badly regressed). F7 reduced to 1.0 (near miss, begin exploiting).

| kappa | Functions | Rationale |
|---|---|---|
| 0.5 | F5, F8 | New best / consistent gains -- exploit hard |
| 1.0 | F7 | Near miss W6 (1.926 vs best 1.958) -- begin exploiting |
| 2.5 | F2, F3, F4, F6 | Stalled or regressed -- explore new regions |
| N/A | F1 | Max-distance sweep -- 6 rounds, no signal |

In [ ]:
# CHANGED: F6 moved from 2.0 -> 2.5 (badly regressed W6); F7 moved from 2.5 -> 1.0 (near miss W6)
kappa_per_function = {
    1: None,
    2: 2.5,
    3: 2.5,
    4: 2.5,
    5: 0.5,
    6: 2.5,   # CHANGED from 2.0
    7: 1.0,   # CHANGED from 2.5
    8: 0.5,
}

strategy_notes = {
    1: "Max-distance sweep (6 rounds, no positive signal)",
    2: "GP-UCB kappa=2.5 (closest yet 0.563 in W6 but still below init best 0.611)",
    3: "GP-UCB kappa=2.5 (best still -0.011 from R2, explore new region)",
    4: "GP-UCB kappa=2.5 (badly regressed W6, searching for R4 basin)",
    5: "GP-UCB kappa=0.5 (marginal new best 6210 W6, maintain exploitation)",
    6: "GP-UCB kappa=2.5 (badly regressed W6 to -1.499, explore new region)",
    7: "GP-UCB kappa=1.0 (near miss 1.926 vs best 1.958 in W6, begin exploiting)",
    8: "GP-UCB kappa=0.5 (new best 9.954 in W6, exploit hard)",
}
print('Kappa assignments set.')

## 6. Run Week 7 Queries

In [ ]:
results = {}

for func_id, d in functions.items():
    X, y = d['X'], d['y']
    n_dims = X.shape[1]
    kappa = kappa_per_function[func_id]

    if func_id == 1:
        best_x = max_distance_query(X)
        gp = fit_gp(X, y)
        mu_b = float(gp.predict(best_x.reshape(1, -1)).ravel()[0])
        sig_b = float(gp.predict(best_x.reshape(1, -1), return_std=True)[1].ravel()[0])
    else:
        best_x, mu_b, sig_b = suggest_next_query(X, y, n_dims, kappa=kappa)

    query_str = format_query(best_x)
    results[func_id] = {
        'query_string': query_str, 'best_x': best_x,
        'current_best_y': float(y.max()), 'predicted_mu': mu_b,
        'predicted_sigma': sig_b, 'kappa': kappa, 'strategy': strategy_notes[func_id],
    }

    print(f"F{func_id} ({n_dims}D) | n={len(y)} | Best Y: {y.max():.6f}")
    print(f"  Strategy : {strategy_notes[func_id]}")
    print(f"  GP mu={mu_b:.4f} | sigma={sig_b:.4f}")
    print(f"  Query    : {query_str}")
    print()

## 7. Submission Strings

In [ ]:
print('=' * 65)
print('WEEK 7 SUBMISSION STRINGS')
print('=' * 65)
for func_id, r in results.items():
    print(f'Function {func_id}: {r["query_string"]}')
print('=' * 65)

## 8. Cumulative Progress Tracker

In [ ]:
all_outputs = [week1_outputs, week2_outputs, week3_outputs,
               week4_outputs, week5_outputs, week6_outputs]
init_bests = {i: np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy').max() for i in range(1,9)}

rows = []
for i in range(1, 9):
    running = init_bests[i]
    bests = [running]
    for w in all_outputs:
        running = max(running, w[i])
        bests.append(running)
    rows.append({
        'Fn': f'F{i}', 'Dims': functions[i]['X'].shape[1], 'n': len(functions[i]['y']),
        'Init': round(bests[0],4), 'R1': round(bests[1],4), 'R2': round(bests[2],4),
        'R3': round(bests[3],4), 'R4': round(bests[4],4), 'R5': round(bests[5],4),
        'R6': round(bests[6],4), 'kappa W7': kappa_per_function[i],
        'W7 Query': results[i]['query_string']
    })

df = pd.DataFrame(rows).set_index('Fn')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 320)
print(df.to_string())

## 9. GP Surface Plots -- 2D Functions (F1 and F2)

In [ ]:
def plot_2d_gp(func_id, X, y, query_x, kappa=2.0, resolution=80, label='Week 7 query'):
    x1 = np.linspace(0, 1, resolution)
    x2 = np.linspace(0, 1, resolution)
    X1, X2 = np.meshgrid(x1, x2)
    grid = np.column_stack([X1.ravel(), X2.ravel()])
    gp = fit_gp(X, y)
    mu, sigma = gp.predict(grid, return_std=True)
    ucb = mu + (kappa or 2.0) * sigma
    n_init = len(np.load(f'{DATA_PATH}function_{func_id}_initial_inputs.npy'))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (data, title, cmap) in zip(axes, [
        (mu.reshape(resolution, resolution),    'GP Mean',         'RdYlGn'),
        (sigma.reshape(resolution, resolution), 'GP Uncertainty',  'Blues'),
        (ucb.reshape(resolution, resolution),   f'UCB (k={kappa})', 'plasma'),
    ]):
        im = ax.contourf(X1, X2, data, levels=30, cmap=cmap)
        plt.colorbar(im, ax=ax)
        ax.scatter(X[:n_init,0], X[:n_init,1], c='white', s=35,
                   edgecolors='k', linewidths=0.8, zorder=5, label='Initial')
        colours = ['yellow','orange','cyan','lime','magenta','aqua']
        markers = ['D','s','^','P','X','v']
        for wk, (col, mrk) in enumerate(zip(colours, markers)):
            idx = n_init + wk
            if idx < len(X) - 1:
                ax.scatter(X[idx,0], X[idx,1], c=col, s=55, marker=mrk,
                           edgecolors='k', linewidths=0.8, zorder=6, label=f'W{wk+1}')
        ax.scatter(query_x[0], query_x[1], c='red', s=160, marker='*',
                   edgecolors='k', linewidths=0.8, zorder=7, label=label)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('x1'); ax.set_ylabel('x2')
        ax.legend(fontsize=6)

    plt.suptitle(f'Function {func_id} -- Week 7 GP Surfaces', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'function_{func_id}_week7_gp.png', dpi=150, bbox_inches='tight')
    plt.show()

for func_id in [1, 2]:
    lbl = 'Max-dist (W7)' if func_id == 1 else 'GP-UCB (W7)'
    plot_2d_gp(func_id, functions[func_id]['X'], functions[func_id]['y'],
               results[func_id]['best_x'], kappa=kappa_per_function[func_id], label=lbl)

## 10. How to Update for Week 8

```python
week7_queries = {
    1: np.array([0.660553, 0.429899]),
    2: np.array([0.594914, 0.176260]),
    3: np.array([0.696015, 0.674845, 0.656797]),
    4: np.array([0.346593, 0.326325, 0.369944, 0.337553]),
    5: np.array([0.943597, 0.976718, 0.948137, 0.999531]),
    6: np.array([0.398706, 0.419380, 0.631066, 0.991715, 0.106696]),
    7: np.array([0.098466, 0.251391, 0.280505, 0.082460, 0.342315, 0.610119]),
    8: np.array([0.115531, 0.115602, 0.041889, 0.228086, 0.497790, 0.351216, 0.169864, 0.523432])
}
week7_outputs = {
    1: <value>, 2: <value>, 3: <value>, 4: <value>,
    5: <value>, 6: <value>, 7: <value>, 8: <value>
}

for i in range(1, 9):
    functions[i]['X'] = np.vstack([functions[i]['X'], week7_queries[i]])
    functions[i]['y'] = np.append(functions[i]['y'], week7_outputs[i])
```